# PBMC 1k 聚类：GitHub 版本


`归一化表达矩阵 → PCA → KNN 图 → Leiden → UMAP`

主图使用 PC15；同时比较 PC15、20、30、50。PC15 只是待验证的候选参数，不是普适规则。

UMAP 仅用于展示。Leiden 簇还需结合 marker、QC 和稳定性检查。


In [ ]:
# 导入依赖并设置临时缓存。
import os
import tempfile
from pathlib import Path

os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
os.environ.setdefault(
    "NUMBA_CACHE_DIR",
    str(Path(tempfile.gettempdir()) / "scanpy_numba_cache"),
)

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from IPython.display import display



In [ ]:
# 集中设置可调参数；这里不写入任何本机绝对路径。
PROJECT_DIR = Path("path/to/your/project")
INPUT_NAME = "pbmc_1k_feature_selection.h5ad"
OUTPUT_NAME = "pbmc_1k_clustering_pc15.h5ad"
FIGURE_DIR_NAME = "clustering_figures_pc15"

N_PCA_COMPS = 50
N_PCS_FOR_GRAPH = 15
N_NEIGHBORS = 15
RESOLUTIONS = [0.25, 0.5, 1.0]
PC_SUMMARY_VALUES = [15, 20, 30, 50]
RANDOM_STATE = 42
WORKING_RESOLUTION = 0.5


In [ ]:
# 根据实际项目结构组装输入和输出位置。
PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
INPUT_PATH = PROJECT_DIR / "results" / INPUT_NAME
OUTPUT_PATH = PROJECT_DIR / "results" / OUTPUT_NAME
FIGURE_DIR = PROJECT_DIR / "results" / FIGURE_DIR_NAME
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## 1. 读取并审计上游对象


In [ ]:
# 读取预处理对象。
adata = ad.read_h5ad(INPUT_PATH)
print(adata)



## 2. 设置聚类表达矩阵

raw counts 保留用于 count-based 分析。PCA 和邻居图使用 scran 归一化表达及选出的 4000 个 deviance 特征。


In [ ]:
# 使用归一化矩阵，保留原始 counts。
adata.X = adata.layers["scran_normalization"].copy()
selected_mask = adata.var["highly_deviant"].to_numpy(dtype=bool)


## 3. PCA 与 PC elbow

先计算 50 个 PC。主图使用 PC15，并在 elbow 表中查看 PC15、20、30、50 的累计方差。

修改 `sc.pp.neighbors` 的 `n_pcs` 会改变邻居图及所有下游结果。保留 `n_comps=50`，可以支持主图参数。


In [ ]:
# 计算 PC 并检查 elbow。
sc.pp.pca(
    adata,
    n_comps=N_PCA_COMPS,
    svd_solver="arpack",
    mask_var="highly_deviant",
    random_state=RANDOM_STATE,
)

sc.pl.pca_variance_ratio(adata, n_pcs=N_PCA_COMPS, log=True)

variance_ratio = adata.uns["pca"]["variance_ratio"]
pc_summary = pd.DataFrame({
    "n_pcs": PC_SUMMARY_VALUES,
    "cumulative_variance": [
        float(variance_ratio[:pcs].sum()) for pcs in PC_SUMMARY_VALUES
    ],
})
display(pc_summary)


## 4. 主 KNN 图、UMAP 与 Leiden

主分析使用 PC15、15 个邻居和固定的 Leiden 参数。UMAP 根据主图计算，但不参与定义聚类。


In [ ]:
# 构建 PC15 邻居图并计算 UMAP。
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS_FOR_GRAPH,
    random_state=RANDOM_STATE,
)
sc.tl.umap(
    adata,
    min_dist=0.3,
    random_state=RANDOM_STATE,
)

print("main neighbors:", adata.uns["neighbors"]["params"])
print("obsp:", list(adata.obsp.keys()))
print("obsm:", list(adata.obsm.keys()))
sc.pl.umap(adata, color=["total_counts", "pct_counts_mt"], ncols=2)


In [ ]:
# 在主图上运行多个 Leiden resolution。
resolution_keys = {}

for resolution in RESOLUTIONS:
    key = f"leiden_res{str(resolution).replace('.', '_')}"
    sc.tl.leiden(
        adata,
        resolution=resolution,
        key_added=key,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=RANDOM_STATE,
    )
    resolution_keys[resolution] = key
    print(
        f"resolution={resolution}: "
        f"{adata.obs[key].nunique()} clusters"
    )
    print(adata.obs[key].value_counts().sort_index().to_dict())

adata.obs["leiden"] = adata.obs[
    resolution_keys[WORKING_RESOLUTION]
].astype("category")

print("provisional working label:", resolution_keys[WORKING_RESOLUTION])


In [ ]:
# 比较不同 resolution；图形分离不等于生物学差异。
sc.pl.umap(
    adata,
    color=[resolution_keys[r] for r in RESOLUTIONS],
    legend_loc="on data",
    ncols=3,
)


## 5. Marker 排名与经典 marker 审计

marker 检验仅用于排序和解释。由于聚类和 marker 检验使用同一批数据，marker 的 p 值不能单独证明某个簇是独立群体。


In [ ]:
# 排序候选 marker 并导出结果。
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    method="wilcoxon",
    use_raw=False,
    layer="scran_normalization",
)

marker_table = sc.get.rank_genes_groups_df(adata, group=None)
symbol_map = adata.var["gene_symbol"].to_dict() if "gene_symbol" in adata.var else {}
marker_table["gene_symbol"] = marker_table["names"].map(symbol_map)
marker_table["gene_symbol"] = marker_table["gene_symbol"].fillna(marker_table["names"])

top_markers = (
    marker_table
    .groupby("group", observed=True)
    .head(12)
    .reset_index(drop=True)
)
display(top_markers[["group", "names", "gene_symbol", "scores", "pvals_adj"]])

marker_table.to_csv(
    OUTPUT_PATH.with_name("pbmc_1k_clustering_pc15_markers.tsv"),
    sep="	",
    index=False,
)


In [ ]:
# 用经典 PBMC marker 检查细胞谱系。
marker_panel = {
    "T_cell": ["CD3D", "TRAC", "IL7R", "TCF7"],
    "B_cell": ["MS4A1", "CD79A", "CD74", "CD37"],
    "NK": ["NKG7", "GNLY", "KLRD1", "PRF1"],
    "Cytotoxic": ["CCL5", "GZMK", "GZMB", "CTSW"],
    "Monocyte": ["LYZ", "S100A8", "S100A9", "FCGR3A"],
    "APC": ["HLA-DRA", "CD74", "HLA-DPA1", "FCER1A"],
    "Plasma": ["JCHAIN", "MZB1", "TNFRSF17"],
    "Platelet": ["PPBP", "PF4", "TUBB1"],
}

sc.pl.dotplot(
    adata,
    marker_panel,
    groupby="leiden",
    layer="scran_normalization",
    gene_symbols="gene_symbol",
    standard_scale="var",
)


## 6. 保存可追溯结果

provenance 会记录实际 PCA 维度和主图使用的 PC 数。保存的对象包含全部 resolution 标签、主 PC15 UMAP 和 marker 排名。


In [ ]:
# 保存准确 provenance、图像和 AnnData。
adata.uns["clustering_provenance"] = {
    "source": "sc-best-practices Chapter 12: Clustering",
    "input_file_name": INPUT_NAME,
    "working_layer": "scran_normalization",
    "feature_mask": "highly_deviant",
    "n_selected_genes": int(selected_mask.sum()),
    "pca_n_comps": int(N_PCA_COMPS),
    "neighbors_n_neighbors": int(N_NEIGHBORS),
    "neighbors_n_pcs_main": int(N_PCS_FOR_GRAPH),
    "umap_min_dist": 0.3,
    "leiden_resolutions": [float(r) for r in RESOLUTIONS],
    "working_resolution_provisional": float(WORKING_RESOLUTION),
    "leiden_flavor": "igraph",
    "leiden_n_iterations": 2,
    "leiden_directed": False,
    "random_state": int(RANDOM_STATE),
}

for resolution, key in resolution_keys.items():
    plt.close("all")
    sc.pl.umap(
        adata,
        color=key,
        legend_loc="on data",
        show=False,
    )
    plt.savefig(
        FIGURE_DIR / f"umap_{key}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")

adata.write_h5ad(OUTPUT_PATH)

print("saved AnnData:", OUTPUT_PATH)
print("saved figures:", FIGURE_DIR)
